# plot_03a. Forced response figures: the mean and signal to noise

The figures for the forced response, drawn from what notebooks calc_02a to calc_02c saved, so nothing slow runs here. Those notebooks explain each method, step by step. The figures are in four notebooks, so that none grows too big to open; each opens only the saved results it uses, so they can be run in any order after notebooks calc_02a to calc_02c:

| Notebook | Sections |
|---|---|
| plot_03a | 1. Mean change; 2. Signal to noise (and a check of every model's headline numbers) |
| plot_03b | 3. Quantile response; 4. Change in width, and which tail |
| plot_03c | 5. The whole distribution; 6. Additivity; 7. Zonal means and the sea-ice edge |
| plot_03d | 8. Probability ratios; 9. The shift-only ratio |

The sections keep one numbering across the four.

Every section shows its result in the same four figures, with the four seasons across every map:

1. **What goes in.** The quantities the result is made from, then the result, one row each: one model and experiment (`EXAMPLE_MODEL`, `EXPERIMENT`).
2. **One model.** The parts and the result for every experiment of `EXAMPLE_MODEL`.
3. **Every model.** The result for every model (rows), one figure per experiment, dotted where it is not significant.
4. **How many models.** The multi-model median of the result (top row), then at each grid point how many models have a significant increase and how many a significant decrease; one figure per experiment.

This notebook:

| Section | Result | Test |
|---|---|---|
| 1. Mean change | final 21 years against hist-nat over the same years | hist-nat bootstrap |
| 2. Signal to noise | \|S/N\|: how far apart the two distributions have moved | none: a measure, not a test |

**By SAM phase.** Section 1 ends with the result in the seasons of each phase of the Southern Annular Mode alone (negative, neutral and positive, cut at hist-nat's terciles of the index: notebook calc_02a, *SAM Phases*), the phases as rows.

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, and dots mark where a change is not significant. The black lines on the maps of section 1 are the sea-ice edge (the 15% contour of the ensemble-mean concentration over the final 21 years, from notebook calc_04): hist-nat's dashed, the experiment's solid; the multi-model figures draw the edge of the models' median concentration. Without the sea ice they are left out. In the counts, the colour changes where more than half the models agree: reds turn brown (an increase) and blues purple (a decrease). The paper figures, which put the models together, are in notebook plot_05. The cells before each figure pick out its data and show them. Each figure is then drawn layer by layer with `extant/plots/antarctica.py`: `map_figure_size` and `map_axes` make the figure, its maps and colour bars, then each layer (a field, its dots, each ice edge) is drawn on those axes by its own call, a row of a different kind being a slice of them (`axes[2]`).

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebooks calc_02a to calc_02c. Until 14 October 2026 the four were one notebook, `04_forced_response_figures.ipynb` (in the git history); the version before that, from 7 October, is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from extant import config, sam, sea_ice
from extant.config import ALPHA, FORCED_EXPERIMENTS, LAST_FULL_YEAR, WINDOW
from extant import significance as sig
from extant.logger import setup_logging
from extant.plots import antarctica, charts, forced_response as fr, paper_panels, sea_ice as sip
from plotting_modules.constants import FORCING_COLORS, SEASONS

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)   # every model, experiment and season in one table

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`ALPHA`, `FORCED_EXPERIMENTS`, `LAST_FULL_YEAR`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [ ]:
VARIABLE = config.VARIABLES['tas']

#(t): The model and experiment of the "what goes in" and "one model" figures
EXAMPLE_MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']

#(t): The |S/N| levels section 2 counts the models above
SN_LEVELS = [0.5, 1, 2]

#(t): The SAM phases of 1.5, in the order of their rows, and how the rows are labelled
PHASES = list(sam.PHASES)
PHASE_LABELS = {phase: f'SAM\n{phase}' for phase in PHASES}

#(t): The share of the models that must have a significant change of one sign for a multi-model change to be robust
ROBUST = config.SIGNIFICANT_THRESHOLD


In [ ]:
#(t): Where notebooks calc_02a to calc_02c and calc_04 saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks plot_04 and plot_05), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook plot_05 (*Check the Colour Ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)

In [ ]:
#(t): The sea-ice zones (notebook calc_04, section 7), coloured as in notebook plot_02
zone_scale = fr.class_scale(sea_ice.ZONES, ['#f4f3ef', '#2c7fb8', '#e6550d', '#31a354', '#bdbdbd'], 'Zone')

In [ ]:
#(t): The legend entries of the maps: the dots, and the two ice edges (hist-nat's dashed, the experiment's solid)
not_significant_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                                    label=f'Not significant (p ≥ {ALPHA:g})')
hist_nat_edge_handle = plt.Line2D([], [], color='k', linestyle='dashed', label="hist-nat's ice edge")
experiment_edge_handle = plt.Line2D([], [], color='k', label="The experiment's ice edge")

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the Saved Results

Run **one** of the next two sections. Both leave `RESULTS_DIR` and `ICE_RESULTS_DIR`, the folders notebooks calc_02a to calc_02c's and notebook calc_04's results are opened from.

### Sample data

In [ ]:
#(t): The sample's results: notebooks calc_02a to calc_02c's, and calc_04's sea ice
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'sample'
RESULTS_DIR, ICE_RESULTS_DIR

### Full data

In [ ]:
#(t): The full data's results: notebooks calc_02a to calc_02c's, and calc_04's sea ice
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'full'
RESULTS_DIR, ICE_RESULTS_DIR

### Saved results

Everything sections 1 and 2 use, and the check of the results below, from `RESULTS_DIR`, one store per cell, loaded into memory.

In [ ]:
#(t): The hist-nat bootstrap of the mean change, with hist-nat's and every experiment's final-years mean
mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'mean_bootstrap.zarr').load()

In [ ]:
#(t): Each experiment's forced response
lesfmip_forced_ds = xr.open_zarr(RESULTS_DIR / 'forced.zarr').load()

In [ ]:
#(t): Each experiment's noise
lesfmip_experiment_noise_ds = xr.open_zarr(RESULTS_DIR / 'experiment_noise.zarr').load()

In [ ]:
#(t): The signal: each experiment's forced response minus hist-nat's
lesfmip_signal_ds = xr.open_zarr(RESULTS_DIR / 'signal.zarr').load()

In [ ]:
#(t): The noise: each experiment's and hist-nat's pooled
lesfmip_noise_ds = xr.open_zarr(RESULTS_DIR / 'noise.zarr').load()

In [ ]:
#(t): S/N
lesfmip_seasonal_sn_ds = xr.open_zarr(RESULTS_DIR / 'sn.zarr').load()

In [ ]:
#(t): The change in Q05, Q50 and Q95
quantile_response_da = xr.open_zarr(RESULTS_DIR / 'quantile_response.zarr')['quantile_response'].load()

In [ ]:
#(t): The width test: the change in Q95 − Q05, and its p-value
qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'qrange_change.zarr').load()

In [ ]:
#(t): How often each experiment is in each SAM phase
sam_phase_share_da = xr.open_zarr(RESULTS_DIR / 'sam_phase_share.zarr')['phase_share'].load()

In [ ]:
#(t): The mean change in each SAM phase
sam_mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'sam_mean_bootstrap.zarr').load()

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = mean_bootstrap_ds.sizes['model']
N_MODELS

### Sea ice

The sea-ice edge on the maps of section 1, and the sea-ice results of 1.6, come from notebook calc_04. Run **one** of the next two sections. Both leave `ice_climatology_da` (the ensemble-mean concentration over the final years), and `model_ice_da` and `median_ice_da` (`EXAMPLE_MODEL`'s concentration, and the models' median, for the edges of the one-model and multi-model maps); without sea ice they are `None` and the figures leave the edge out. *With sea ice* also opens the zones and the changes 1.6 uses; without it, 1.6 cannot run.

#### Without sea ice

Before notebook calc_04 has saved the sea ice (on the sample, that is when the notebooks are run in order).

In [ ]:
ice_climatology_da = model_ice_da = median_ice_da = None

#### With sea ice

In [ ]:
#(t): The ensemble-mean concentration over the final years (%), EXAMPLE_MODEL's, and the models' median (for the edges)
ice_climatology_da = xr.open_zarr(ICE_RESULTS_DIR / 'climatology.zarr')['siconc'].load()
model_ice_da = ice_climatology_da.sel(model=EXAMPLE_MODEL)
median_ice_da = ice_climatology_da.median('model')

In [ ]:
#(t): Each cell's zone in the final years (the numbers of sea_ice.ZONES), every model, experiment and season
ice_zones_da = xr.open_zarr(ICE_RESULTS_DIR / 'zones.zarr')['zone'].load()

In [ ]:
#(t): The temperature changes averaged over each zone, and each zone's area (million km²)
tas_zone_means_ds = xr.open_zarr(ICE_RESULTS_DIR / 'zone_means.zarr').load()

In [ ]:
#(t): The change in extent (million km²) and in the edge's latitude (degrees) against hist-nat
extent_change_ds = xr.open_zarr(ICE_RESULTS_DIR / 'extent_change.zarr').load()

In [ ]:
#(t): The temperature changes averaged over the cells with ice in hist-nat's final years
ice_region_change_ds = xr.open_zarr(ICE_RESULTS_DIR / 'ice_region_change.zarr').load()

## Check the Results

The headline changes of every model, experiment and season, averaged over the whole area (cos-latitude weighted). A model whose numbers sit far from the others' has probably gone wrong before notebooks calc_02a to calc_02c (in its raw files or their conversion), and every figure below would show it; look at its members in notebook calc_02a (*What was opened*) before reading anything into it.

In [ ]:
#(t): The headline changes side by side: the mean, three quantiles, the width, and S/N in the centre of the final years
headline_ds = xr.Dataset({
    'mean change': mean_bootstrap_ds['mean_change'],
    'ΔQ05': quantile_response_da.sel(quantile=0.05, drop=True),
    'ΔQ50': quantile_response_da.sel(quantile=0.50, drop=True),
    'ΔQ95': quantile_response_da.sel(quantile=0.95, drop=True),
    'Δ(Q95 − Q05)': qrange_change_ds['qrange_change'],
    'S/N': lesfmip_seasonal_sn_ds[VARIABLE.name].isel(year=LAST_FULL_YEAR, drop=True),
})
headline_ds = headline_ds.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Their area means, one row per model, experiment and season
headline_area_ds = sig.area_mean(headline_ds)
headline_area_df = headline_area_ds.to_dataframe().round(2)
headline_area_df

## 1. Mean Change

The mean of each experiment's final 21 years minus hist-nat's over the same years, members and years pooled (notebook calc_02b, section 1):

$$
\Delta\bar{x} = \bar{x}_{\mathrm{exp}} - \bar{x}_{\mathrm{nat}}.
$$

It is tested with the hist-nat bootstrap: the experiment's mean against the means of 1000 trials, each 10 random hist-nat members over the same 21 years, so natural variability, the correlation between a member's years included, sets what counts as significant. The change is significant where $p < 0.05$. (Notebook calc_02b also saves the two Welch t-tests, `ttest` and `ttest_rolling_median`, which count correlated values as independent and so call more changes significant, and the member-block permutation test, `mean_block`.)

### 1.1 What goes in

hist-nat's mean and `EXPERIMENT`'s, both over the final 21 years, for `EXAMPLE_MODEL`; the change between them; and the test's p-value.

In [ ]:
#(t): hist-nat's mean over the final WINDOW years, EXAMPLE_MODEL, every season
hist_nat_mean_da = mean_bootstrap_ds['hist_nat_mean'].sel(model=EXAMPLE_MODEL, season=SEASONS)

In [ ]:
#(t): EXPERIMENT's over the same years
experiment_mean_da = mean_bootstrap_ds['experiment_mean'].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                              season=SEASONS)

In [ ]:
#(t): The change (mean_bootstrap_ds['mean_change'] holds the same)
mean_change_da = experiment_mean_da - hist_nat_mean_da
mean_change_da

In [ ]:
#(t): Its p-value, from the hist-nat bootstrap
mean_pvalue_da = mean_bootstrap_ds['pvalue'].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)

In [ ]:
#(t): Not significant: p >= ALPHA (False where there is no data)
mean_not_significant_da = mean_pvalue_da >= ALPHA
mean_not_significant_da

In [ ]:
#(t): What goes in: hist-nat's and EXPERIMENT's mean (one colour bar), the change, dotted where not significant and
#(t): with both ice edges, and the p-value
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=4, n_cols=len(SEASONS), n_colorbar_rows=3, n_legends=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=4, n_cols=len(SEASONS))
#(c): Each colour bar under the rows it is for: the means' under rows 1 and 2, the change's under row 3, the
#(c): p-value's under row 4
mean_cax = antarctica.colorbar_axis_below(axes[:2])
change_cax = antarctica.colorbar_axis_below(axes[2])
pvalue_cax = antarctica.colorbar_axis_below(axes[3])

#(c): Rows 1 and 2: hist-nat's and EXPERIMENT's mean, on one colour bar
antarctica.heatmaps_of_antarctica(hist_nat_mean_da, None, 'season', axes=axes[0], cax=mean_cax,
                                  **scales.temperature.colours())
antarctica.heatmaps_of_antarctica(experiment_mean_da, None, 'season', axes=axes[1], **scales.temperature.colours())
#(c): Row 3: the change, its dots, and the ice edges (if the sea ice was opened)
antarctica.heatmaps_of_antarctica(mean_change_da, None, 'season', axes=axes[2], cax=change_cax,
                                  **scales.change.colours())
antarctica.not_significant_dots_grid(mean_not_significant_da, None, 'season', axes=axes[2])
if model_ice_da is not None:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season', axes=axes[2],
                         linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT, season=SEASONS), None, 'season', axes=axes[2])
#(c): Row 4: the p-value
antarctica.heatmaps_of_antarctica(mean_pvalue_da, None, 'season', axes=axes[3], cax=pvalue_cax,
                                  **scales.pvalue.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [f'hist-nat\nfinal {WINDOW} years', f'{EXPERIMENT}\nfinal {WINDOW} years',
                             f'Change\n{EXPERIMENT} − hist-nat', 'p-value\nhist-nat bootstrap'])
fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center',
           ncols=3, fontsize=8)
fig.suptitle(f'What goes into the mean change: {EXAMPLE_MODEL}, {EXPERIMENT}');

### 1.2 One model

The change for every experiment of `EXAMPLE_MODEL`, dotted where it is not significant (1.1 shows the p-values).

In [ ]:
#(t): EXAMPLE_MODEL's change in every experiment and season
model_mean_change_da = mean_bootstrap_ds['mean_change'].sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS,
                                                            season=SEASONS)
model_mean_change_da

In [ ]:
#(t): Its p-value
model_mean_pvalue_da = mean_bootstrap_ds['pvalue'].sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS,
                                                       season=SEASONS)

In [ ]:
#(t): Not significant
model_mean_not_significant_da = model_mean_pvalue_da >= ALPHA
model_mean_not_significant_da

In [ ]:
#(t): EXAMPLE_MODEL's mean change in every experiment (rows), dotted where not significant, with hist-nat's ice edge
#(t): (dashed)
#(t): and each experiment's (solid)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_mean_change_da, 'experiment', 'season', axes=axes, cax=cax,
                                  **scales.change.colours())
antarctica.not_significant_dots_grid(model_mean_not_significant_da, 'experiment', 'season', axes=axes)
#(c): The ice edges, if the sea ice was opened: hist-nat's on every row (given the rows' experiments), each experiment's
if model_ice_da is not None:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS,
                                          drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                         'experiment', 'season', axes=axes, linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS), 'experiment', 'season',
                         axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center',
           ncols=3, fontsize=8)
fig.suptitle(f'{EXAMPLE_MODEL}: mean change, final {WINDOW} years against hist-nat');

### 1.3 Every model

One figure per experiment: every model down, every season across.

In [ ]:
#(t): The change for every model, experiment and season
all_mean_change_da = mean_bootstrap_ds['mean_change'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Significant: p < ALPHA (False where there is no data)
all_mean_significant_da = mean_bootstrap_ds['pvalue'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS) < ALPHA

In [ ]:
#(t): Not significant, where there is data
all_mean_not_significant_da = ~all_mean_significant_da & all_mean_change_da.notnull()
all_mean_not_significant_da

In [ ]:
#(t): Every model (rows), one figure per experiment, dotted where not significant, with each model's ice edges
models = all_mean_change_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_mean_change_da.sel(experiment=experiment), 'model', 'season', axes=axes,
                                      cax=cax, **scales.change.colours())
    antarctica.not_significant_dots_grid(all_mean_not_significant_da.sel(experiment=experiment), 'model', 'season',
                                         axes=axes)
    #(c): The ice edges in the rows' order (empty for a model without sea ice), if the sea ice was opened
    if ice_climatology_da is not None:
        rows_ice_da = ice_climatology_da.reindex(model=models).sel(season=SEASONS)
        antarctica.ice_edges(rows_ice_da.sel(experiment='hist-nat'), 'model', 'season', axes=axes, linestyle='dashed')
        antarctica.ice_edges(rows_ice_da.sel(experiment=experiment), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle],
               loc='outside lower center', ncols=3,
               fontsize=8)
    fig.suptitle(f'Mean change, final {WINDOW} years against hist-nat, every model: {experiment}')

### 1.4 How many models

The multi-model median of the change (top row), then at each grid point the number of models whose mean change is significant and positive, and significant and negative. One figure per experiment.

In [ ]:
#(t): The multi-model median of the change (a median, so one model's bad data cannot drag it)
median_mean_change_da = all_mean_change_da.median('model')
median_mean_change_da

In [ ]:
#(t): How many models have a significant increase, and how many a significant decrease
mean_counts_da = sig.significant_counts(all_mean_change_da, all_mean_significant_da)

In [ ]:
#(t): The decreases as negative counts, so that one colour bar (blues for decreases, reds for increases) serves both
#(t): rows of counts
signed_mean_counts_da = xr.concat([mean_counts_da.sel(direction=['increase']),
                                   -mean_counts_da.sel(direction=['decrease'])], dim='direction')
signed_mean_counts_da

In [ ]:
#(t): Their colours: blues and reds up to half the models, purples and browns beyond, white for none
mean_count_colours = fr.count_scale(N_MODELS, f'Number of models with a significant decrease or increase, '
                                              f'p < {ALPHA:g}').colours()

In [ ]:
#(t): The multi-model median (top row), then how many models have a significant increase and a significant decrease,
#(t): one figure per experiment, with the ice edges of the models' median concentration
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
    #(c): The median's colour bar under row 1, the counts' under rows 2 and 3
    median_cax = antarctica.colorbar_axis_below(axes[0])
    count_cax = antarctica.colorbar_axis_below(axes[1:])

    antarctica.heatmaps_of_antarctica(median_mean_change_da.sel(experiment=experiment), None, 'season', axes=axes[0],
                                      cax=median_cax, **scales.change.colours())
    antarctica.heatmaps_of_antarctica(signed_mean_counts_da.sel(experiment=experiment), 'direction', 'season',
                                      axes=axes[1:], cax=count_cax, **mean_count_colours)
    #(c): Both ice edges on every row, if the sea ice was opened
    if median_ice_da is not None:
        for row_axes in axes:
            antarctica.ice_edges(median_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season',
                                 axes=row_axes, linestyle='dashed')
            antarctica.ice_edges(median_ice_da.sel(experiment=experiment, season=SEASONS), None, 'season',
                                 axes=row_axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, ['Multi-model\nmedian', 'Significant\nincrease', 'Significant\ndecrease'])
    fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
    fig.suptitle(f'How many models have a significant change in the mean: {experiment}')

### 1.5 By SAM phase

The mean change in the seasons of each SAM phase alone (notebooks calc_02a, *SAM Phases*, and calc_02b, section 1, *By SAM phase*): each experiment's seasons in a phase against hist-nat's in the same phase, the phases cut at hist-nat's terciles of the SAM index (so a third of hist-nat's seasons fall in each). First how often each experiment is in each phase; then the change in each phase for `EXAMPLE_MODEL`, dotted where it is not significant; the multi-model median, dotted where it is not robust (fewer than two-thirds of the models significant with one sign); and how many models are significant with each sign.

In [ ]:
#(t): How often each experiment of EXAMPLE_MODEL is in each phase over the final WINDOW years
model_phase_share_da = sam_phase_share_da.sel(model=EXAMPLE_MODEL, season=SEASONS)
model_phase_share_da

In [ ]:
model_phase_share_da.to_series().unstack('season')[SEASONS].round(2)

In [ ]:
#(t): The share of the seasons in the positive phase, every model and experiment
sam_phase_share_da.sel(phase='positive', season=SEASONS).to_series().unstack('season')[SEASONS].round(2)

In [ ]:
#(t): EXAMPLE_MODEL's change in the mean in each phase, every experiment and season
model_phase_mean_change_da = sam_mean_bootstrap_ds['mean_change'].sel(model=EXAMPLE_MODEL,
                                                                      experiment=FORCED_EXPERIMENTS, season=SEASONS)
model_phase_mean_change_da

In [ ]:
#(t): Not significant: p >= ALPHA (False where there is no data)
model_phase_mean_not_significant_da = sam_mean_bootstrap_ds['pvalue'].sel(model=EXAMPLE_MODEL,
                                                                          experiment=FORCED_EXPERIMENTS,
                                                                          season=SEASONS) >= ALPHA
model_phase_mean_not_significant_da

In [ ]:
#(t): EXAMPLE_MODEL's mean change in each SAM phase (rows), one figure per experiment, dotted where not significant
phases = model_phase_mean_change_da['phase'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(phases), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(phases), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(model_phase_mean_change_da.sel(experiment=experiment), 'phase', 'season', axes=axes,
                                      cax=cax, **scales.change.colours())
    antarctica.not_significant_dots_grid(model_phase_mean_not_significant_da.sel(experiment=experiment), 'phase', 'season',
                                         axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in phases])
    fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=8)
    fig.suptitle(f'{EXAMPLE_MODEL}: mean change in each SAM phase, final {WINDOW} years against hist-nat: {experiment}')

In [ ]:
#(t): The change in each phase, every model, experiment and season
all_phase_mean_change_da = sam_mean_bootstrap_ds['mean_change'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Significant: p < ALPHA (False where there is no data)
all_phase_mean_significant_da = sam_mean_bootstrap_ds['pvalue'].sel(experiment=FORCED_EXPERIMENTS,
                                                                    season=SEASONS) < ALPHA

In [ ]:
#(t): The multi-model median
median_phase_mean_change_da = all_phase_mean_change_da.median('model')
median_phase_mean_change_da

In [ ]:
#(t): Where the models agree: +1 (-1) where at least ROBUST of them have a significant increase (decrease)
phase_mean_robust_da = sig.robust_sign(all_phase_mean_change_da, all_phase_mean_significant_da, threshold=ROBUST)

In [ ]:
#(t): Not robust, where there is data
phase_mean_not_robust_da = phase_mean_robust_da == 0
phase_mean_not_robust_da

In [ ]:
#(t): The multi-model median in each SAM phase (rows), one figure per experiment, dotted where not robust
phases = median_phase_mean_change_da['phase'].values
not_robust_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                               label='Not robust (fewer than two-thirds\nof the models significant with one sign)')
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(phases), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(phases), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(median_phase_mean_change_da.sel(experiment=experiment), 'phase', 'season',
                                      axes=axes, cax=cax, **scales.change.colours())
    antarctica.not_significant_dots_grid(phase_mean_not_robust_da.sel(experiment=experiment), 'phase', 'season',
                                         axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in phases])
    fig.legend(handles=[not_robust_handle], loc='outside lower center', fontsize=8)
    fig.suptitle(f'Mean change in each SAM phase, multi-model median: {experiment}')

In [ ]:
#(t): How many models have a significant increase, and how many a significant decrease, in each phase
phase_mean_counts_da = sig.significant_counts(all_phase_mean_change_da, all_phase_mean_significant_da)

In [ ]:
#(t): The decreases as negative counts, as in 1.4
signed_phase_mean_counts_da = xr.concat([phase_mean_counts_da.sel(direction=['increase']),
                                         -phase_mean_counts_da.sel(direction=['decrease'])], dim='direction')
signed_phase_mean_counts_da

In [ ]:
#(t): How many models have a significant increase and decrease in each SAM phase: two rows per phase, one colour bar
phases = signed_phase_mean_counts_da['phase'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2 * len(phases), n_cols=len(SEASONS),
                                                        n_colorbar_rows=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2 * len(phases), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    #(c): Each phase's two rows, increase then decrease; the colour bar drawn once, with the first phase
    for k, phase in enumerate(phases):
        antarctica.heatmaps_of_antarctica(signed_phase_mean_counts_da.sel(experiment=experiment, phase=phase),
                                          'direction', 'season', axes=axes[2 * k:2 * k + 2],
                                          cax=cax if k == 0 else None, **mean_count_colours)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [f'SAM {phase}\n{direction}' for phase in phases for direction in ['increase', 'decrease']])
    fig.suptitle(f'How many models have a significant change in the mean in each SAM phase: {experiment}')

### 1.6 Sea ice and the mean change

Where the ice changed, and how the mean changed there (notebook calc_04, section 7). Each ocean cell is sorted by its concentration in hist-nat's final years and the experiment's: open water in both, ice in both, ice lost, ice gained, or the continent. If losing the ice drives the warming, "ice lost" warms more than "open water" and "ice in both", most in autumn and winter. First what goes in for `EXAMPLE_MODEL` and `EXPERIMENT`; then the mean change averaged over each zone, every model; then, one point per model and forcing, the change in extent against the mean change over hist-nat's ice (where ice can be lost). Needs the sea ice (*With sea ice* above).

In [ ]:
#(t): EXAMPLE_MODEL's zones for EXPERIMENT, every season
model_zones_da = ice_zones_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)
model_zones_da

In [ ]:
#(t): What goes in: the zones, and EXAMPLE_MODEL's mean change for EXPERIMENT, each with both ice edges
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(SEASONS))
#(c): Each row's colour bar under it
zone_cax = antarctica.colorbar_axis_below(axes[0])
change_cax = antarctica.colorbar_axis_below(axes[1])

antarctica.heatmaps_of_antarctica(model_zones_da, None, 'season', axes=axes[0], cax=zone_cax, **zone_scale.colours())
antarctica.heatmaps_of_antarctica(mean_bootstrap_ds['mean_change'].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                                       season=SEASONS),
                                  None, 'season', axes=axes[1], cax=change_cax, **scales.change.colours())
#(c): Both ice edges on every row
for row_axes in axes:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season', axes=row_axes,
                         linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT, season=SEASONS), None, 'season', axes=row_axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, ['Zone', f'Δ mean\n{EXPERIMENT} − hist-nat'])
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'Where the ice changed, and the mean: {EXAMPLE_MODEL}, {EXPERIMENT}, final {WINDOW} years');

In [ ]:
#(t): The mean change averaged over each zone, every model, experiment and season (°C)
zone_mean_change_da = tas_zone_means_ds['mean']

In [ ]:
#(t): The legend of the zone figures: each model's marker, and the bar of their median
zone_legend_handles = (charts.model_legend_handles(zone_mean_change_da['model'].values)
                       + [Patch(color='0.6', alpha=0.5, label='Multi-model median')])

In [ ]:
#(t): The mean change in each zone: experiments down, seasons across; a bar for the median of the models and a marker
#(t): for each model, in the experiment's colour
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(zone_mean_change_da.sel(experiment=experiment, season=SEASONS), 'zone', None, 'season',
                           axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ mean (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle(f'Change in the mean temperature in each zone, final {WINDOW} years against hist-nat', x=0.01, ha='left');

In [ ]:
#(t): The change in extent (million km²) and the mean change over hist-nat's ice (°C), every model and experiment
extent_change_da = extent_change_ds['extent_change']
ice_region_mean_change_da = ice_region_change_ds['mean']
ice_region_mean_change_da

In [ ]:
#(t): The change in extent against the mean change over hist-nat's ice: one point per model (marker) and forcing
#(t): (colour), one panel per season; r is the correlation over all the points
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    charts.plot_scatter_by_forcing(extent_change_da.sel(experiment=FORCED_EXPERIMENTS, season=season),
                                         ice_region_mean_change_da.sel(experiment=FORCED_EXPERIMENTS,
                                                                       season=season), ax=ax)
    ax.set_title(season, loc='left')
    ax.set_xlabel('Δ extent (million km²)')
axes[0, 0].set_ylabel("Δ mean over hist-nat's ice (°C)")
legend_axes[0].legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS), loc='center',
                      ncols=len(FORCED_EXPERIMENTS), frameon=False)
legend_axes[1].legend(handles=charts.model_legend_handles(extent_change_da['model'].values),
                      loc='center', ncols=extent_change_da.sizes['model'], frameon=False)
fig.suptitle(f"Change in extent against the mean change over hist-nat's ice, final {WINDOW} years", x=0.01, ha='left');

## 2. Signal to Noise

S/N in the centre of the final 21 years (2003), the last year with a complete noise window, so it covers the same years as the tests (notebook calc_02b, section 2):

- the **signal** $S$ is the experiment's forced response minus hist-nat's, where a forced response is the ensemble mean smoothed with LOWESS (81 years);
- the **noise** $N$ pools the two experiments' standard deviations $\sigma$ of each member about its forced response, over the 21 years centred on that year: $N = \sqrt{\tfrac{1}{2}\left(\sigma^2_{\mathrm{exp}} + \sigma^2_{\mathrm{nat}}\right)}$.

S/N is not a test here. It says how far apart the experiment's and hist-nat's distributions have moved, in units of their year-to-year noise: two normal distributions 0.5, 1 and 2 noise apart share about 80%, 62% and 32% of their area. The maps show $|\mathrm{S/N}|$, whichever way the shift goes (the signal gives its sign): yellow below 1, orange from 1 to 2, red above 2. Section 2.4 counts the models above 0.5, 1 and 2.

### 2.1 What goes in

The forced responses of hist-nat and `EXPERIMENT` and the signal between them, then their noise and the pooled noise; `EXAMPLE_MODEL`.

In [ ]:
#(t): The year shown: the centre of the final WINDOW years
sn_year = int(lesfmip_seasonal_sn_ds['year'][LAST_FULL_YEAR])

In [ ]:
#(t): hist-nat's forced response in that year, EXAMPLE_MODEL, every season
hist_nat_forced_da = lesfmip_forced_ds[VARIABLE.name].sel(model=EXAMPLE_MODEL, experiment='hist-nat', season=SEASONS,
                                                          year=sn_year)

In [ ]:
#(t): EXPERIMENT's
experiment_forced_da = lesfmip_forced_ds[VARIABLE.name].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS,
                                                            year=sn_year)

In [ ]:
#(t): The signal (lesfmip_signal_ds holds the same)
signal_da = experiment_forced_da - hist_nat_forced_da
signal_da

In [ ]:
#(t): What goes into the signal: hist-nat's and EXPERIMENT's forced response (one colour bar), and their difference
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=2),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
#(c): The forced responses' colour bar under rows 1 and 2, the signal's under row 3
forced_cax = antarctica.colorbar_axis_below(axes[:2])
signal_cax = antarctica.colorbar_axis_below(axes[2])

antarctica.heatmaps_of_antarctica(hist_nat_forced_da, None, 'season', axes=axes[0], cax=forced_cax,
                                  **scales.temperature.colours())
antarctica.heatmaps_of_antarctica(experiment_forced_da, None, 'season', axes=axes[1], **scales.temperature.colours())
antarctica.heatmaps_of_antarctica(signal_da, None, 'season', axes=axes[2], cax=signal_cax, **scales.change.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, ['hist-nat\nforced response', f'{EXPERIMENT}\nforced response',
                             f'Signal\n{EXPERIMENT} − hist-nat'])
fig.suptitle(f'What goes into the signal: {EXAMPLE_MODEL}, {EXPERIMENT}, {sn_year}');

In [ ]:
#(t): hist-nat's standard deviation about its forced response, in the WINDOW years centred on sn_year
hist_nat_sigma_da = lesfmip_experiment_noise_ds[VARIABLE.name].sel(model=EXAMPLE_MODEL, experiment='hist-nat',
                                                                   season=SEASONS,
                                                              year=sn_year)

In [ ]:
#(t): EXPERIMENT's
experiment_sigma_da = lesfmip_experiment_noise_ds[VARIABLE.name].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                                     season=SEASONS,
                                                                year=sn_year)

In [ ]:
#(t): The noise: the two pooled (lesfmip_noise_ds holds the same)
noise_da = np.sqrt(0.5 * (hist_nat_sigma_da ** 2 + experiment_sigma_da ** 2))
noise_da

In [ ]:
#(t): What goes into the noise: hist-nat's and EXPERIMENT's standard deviation, and the two pooled (one colour bar)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(hist_nat_sigma_da, None, 'season', axes=axes[0], cax=cax,
                                  **scales.noise.colours())
antarctica.heatmaps_of_antarctica(experiment_sigma_da, None, 'season', axes=axes[1], **scales.noise.colours())
antarctica.heatmaps_of_antarctica(noise_da, None, 'season', axes=axes[2], **scales.noise.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, ['hist-nat\nσ', f'{EXPERIMENT}\nσ', 'Noise\nthe two pooled'])
fig.suptitle(f'What goes into the noise: {EXAMPLE_MODEL}, {EXPERIMENT}, the {WINDOW} years centred on {sn_year}');

### 2.2 One model

The signal, the noise and |S/N| for every experiment of `EXAMPLE_MODEL`.

In [ ]:
#(t): EXAMPLE_MODEL's signal in every experiment and season, sn_year
model_signal_da = lesfmip_signal_ds[VARIABLE.name].sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS,
                                                       season=SEASONS, year=sn_year)
model_signal_da

In [ ]:
#(t): Its noise
model_noise_da = lesfmip_noise_ds[VARIABLE.name].sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS,
                                                     season=SEASONS, year=sn_year)
model_noise_da

In [ ]:
#(t): S/N
model_sn_da = lesfmip_seasonal_sn_ds[VARIABLE.name].sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS,
                                                        season=SEASONS, year=sn_year)

In [ ]:
#(t): |S/N|: how far apart the distributions have moved, whichever way
model_abs_sn_da = np.abs(model_sn_da)
model_abs_sn_da

In [ ]:
#(t): EXAMPLE_MODEL's signal in every experiment (rows)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_signal_da, 'experiment', 'season', axes=axes, cax=cax,
                                  **scales.change.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.suptitle(f'{EXAMPLE_MODEL}: signal, {sn_year}');

In [ ]:
#(t): EXAMPLE_MODEL's noise in every experiment (rows)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_noise_da, 'experiment', 'season', axes=axes, cax=cax,
                                  **scales.noise.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.suptitle(f'{EXAMPLE_MODEL}: noise, {sn_year}');

In [ ]:
#(t): EXAMPLE_MODEL's |S/N| in every experiment (rows)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_abs_sn_da, 'experiment', 'season', axes=axes, cax=cax,
                                  **scales.sn.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.suptitle(f'{EXAMPLE_MODEL}: |S/N|, {sn_year}');

### 2.3 Every model

One figure per experiment: every model down, every season across.

In [ ]:
#(t): S/N for every model, experiment and season, sn_year
all_sn_da = lesfmip_seasonal_sn_ds[VARIABLE.name].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS, year=sn_year)

In [ ]:
#(t): |S/N|
all_abs_sn_da = np.abs(all_sn_da)

In [ ]:
#(t): Every model's |S/N| (rows), one figure per experiment
models = all_abs_sn_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_abs_sn_da.sel(experiment=experiment), 'model', 'season', axes=axes,
                                      cax=cax, **scales.sn.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.suptitle(f'|S/N|, every model: {experiment}, {sn_year}')

### 2.4 How many models

The multi-model median of |S/N| (top row), then the number of models with |S/N| above 0.5, 1 and 2. Grey is up to half the models, green more than half. One figure per experiment.

In [ ]:
#(t): The levels, on a 'level' dim, so every model is compared with each at once
sn_levels_da = xr.DataArray(SN_LEVELS, dims='level', coords={'level': SN_LEVELS})

In [ ]:
#(t): How many models have |S/N| above each level
sn_counts_da = (all_abs_sn_da > sn_levels_da).sum('model')
sn_counts_da

In [ ]:
#(t): The multi-model median of |S/N|
median_abs_sn_da = all_abs_sn_da.median('model')
median_abs_sn_da

In [ ]:
#(t): The colours of the counts with no sign: greys up to half the models, greens beyond, white for none
sn_count_colours = fr.majority_scale(N_MODELS, 'Number of models with |S/N| above the level').colours()

In [ ]:
#(t): The multi-model median of |S/N| (top row), then how many models are above each level, one figure per experiment
levels = sn_counts_da['level'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=1 + len(levels), n_cols=len(SEASONS),
                                                        n_colorbar_rows=2), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=1 + len(levels), n_cols=len(SEASONS))
    #(c): The median's colour bar under row 1, the counts' under the rest
    median_cax = antarctica.colorbar_axis_below(axes[0])
    count_cax = antarctica.colorbar_axis_below(axes[1:])

    antarctica.heatmaps_of_antarctica(median_abs_sn_da.sel(experiment=experiment), None, 'season', axes=axes[0],
                                      cax=median_cax, **scales.sn.colours())
    antarctica.heatmaps_of_antarctica(sn_counts_da.sel(experiment=experiment), 'level', 'season', axes=axes[1:],
                                      cax=count_cax, **sn_count_colours)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, ['Multi-model\nmedian', *[f'|S/N| > {level:g}' for level in levels]])
    fig.suptitle(f'How many models have moved how far: |S/N| of {experiment}, {sn_year}')